In [1]:
# model2_gurobi.py
from gurobipy import Model, GRB, quicksum
import math

# --------------------------
# Example synthetic data
# Replace these with your real CSV reads / data structures
# --------------------------
T = 10
years = list(range(1, T+1))

# Nodes
N = ['node1', 'node2']

# Projects
# Each project can affect one or multiple nodes; K[i][n] = capacity added at node n (MW)
I = ['projA', 'projB', 'projC']
C = {'projA': 1_200_000, 'projB': 2_500_000, 'projC': 800_000}
K = {
    'projA': {'node1': 20.0, 'node2': 0.0},
    'projB': {'node1': 0.0,  'node2': 30.0},
    'projC': {'node1': 10.0, 'node2': 5.0}
}
Cap0 = {'node1': 50.0, 'node2': 40.0}

# Deterministic demand growth per node per year (MW)
D = {}
for n in N:
    for t in years:
        D[(n,t)] = (45.0 if n=='node1' else 35.0) + 3.0*(t-1)  # simple linear growth

# Annual budgets (can be dynamic)
B = {t: 2_000_000 for t in years}  # e.g., 2M budget each year

# Economics
r = 0.05
VOLL = 1_000_000  # high penalty for unmet load (choose large to enforce reliability)

# --------------------------
# Build model
# --------------------------
m = Model("Intertemporal_DSO")
m.setParam('OutputFlag', 1)
m.setParam('TimeLimit', 600)  # optional

# Decision variables
b = {}   # build in year t (binary)
z = {}   # built by end of year t (binary / cumulative)
u = {}   # unmet demand (continuous >=0)

for i in I:
    for t in years:
        b[i,t] = m.addVar(vtype=GRB.BINARY, name=f"b_{i}_{t}")
        z[i,t] = m.addVar(vtype=GRB.BINARY, name=f"z_{i}_{t}")
for n in N:
    for t in years:
        u[n,t] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name=f"u_{n}_{t}")

m.update()

# Constraints

# Link z (cumulative) with b (build-in-year)
for i in I:
    # z_{i,1} = b_{i,1}
    m.addConstr(z[i,1] == b[i,1], name=f"cum_init_{i}")
    for t in years[1:]:
        # z_{i,t} = z_{i,t-1} + b_{i,t}
        m.addConstr(z[i,t] == z[i,t-1] + b[i,t], name=f"cum_link_{i}_{t}")

# Capacity constraints per node-year
for n in N:
    for t in years:
        cap_expr = Cap0[n] + quicksum(K[i][n] * z[i,t] for i in I)
        # cap >= D - u  ->  cap + u >= D
        m.addConstr(cap_expr + u[n,t] >= D[(n,t)], name=f"cap_{n}_{t}")

# Annual budget constraints
for t in years:
    m.addConstr(quicksum(C[i] * b[i,t] for i in I) <= B[t], name=f"budget_{t}")

# Objective: discounted capex + discounted VOLL * unmet
obj_invest = quicksum((C[i] / ((1+r)**(t-1))) * b[i,t] for i in I for t in years)
obj_unmet = quicksum((VOLL / ((1+r)**(t-1))) * u[n,t] for n in N for t in years)
m.setObjective(obj_invest + obj_unmet, GRB.MINIMIZE)

# Solve
m.optimize()

# --------------------------
# Postprocess: print selected builds and capacity timeline
# --------------------------
print("\nBuild schedule (project -> year):")
for i in I:
    built = False
    for t in years:
        if b[i,t].X > 0.5:
            print(f"  {i} built in year {t}")
            built = True
    if not built:
        print(f"  {i}: not built in horizon")

print("\nNode capacity timeline and unmet demand:")
for n in N:
    for t in years:
        cap = Cap0[n] + sum(K[i][n] * z[i,t].X for i in I)
        print(f"  {n}, year {t}: capacity={cap:.1f}, demand={D[(n,t)]:.1f}, unmet={u[n,t].X:.3f}")


Set parameter Username
Set parameter LicenseID to value 2706884
Academic license - for non-commercial use only - expires 2026-09-10
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 600
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[rosetta2] - Darwin 24.6.0 24G90)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 60 rows, 80 columns and 177 nonzeros
Model fingerprint: 0x4e6f18d7
Variable types: 20 continuous, 60 integer (60 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+06]
  Objective range  [5e+05, 2e+06]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]
Found heuristic solution: objective 1.321392e+08
Presolve removed 60 rows and 80 columns
Presolve time: 0.01s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available pr